# Sitting Posture Evaluation — Video Notebook

This notebook processes a **recorded video file** (not a live webcam) using YOLO person detection and MediaPipe Pose. It saves an annotated video and a CSV file of frame-by-frame pose features.

Run the cells from top to bottom. If you see a MediaPipe import/API error, run the installation cell, restart the kernel, and then run the notebook again.

## 1. Install / repair dependencies

Run this cell if the required packages are not installed or if `mediapipe` does not have `solutions`. After installing, use **Kernel → Restart Kernel** in VS Code, then run the notebook from the top.

MediaPipe `0.10.21` supports the `mp.solutions.pose` API used below.

In [1]:
%pip install opencv-python mediapipe==0.10.21 ultralytics pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and environment check

In [1]:
import os
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from ultralytics import YOLO

print('MediaPipe version:', getattr(mp, '__version__', 'unknown'))
print('MediaPipe loaded from:', mp.__file__)
print('Has mp.solutions:', hasattr(mp, 'solutions'))

if not hasattr(mp, 'solutions'):
    raise RuntimeError(
        'This MediaPipe import does not expose mp.solutions. Check that the notebook is using your project .venv, '
        'run the installation cell, restart the kernel, and make sure there is no local file/folder named mediapipe.'
    )

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
print('Imports successful.')

ImportError: DLL load failed while importing _framework_bindings: A dynamic link library (DLL) initialization routine failed.

## 3. Set your video path and output folder

Change `VIDEO_PATH` to the location of your recorded video. The `r` before the string is important on Windows because it treats backslashes as literal characters.

For example: `r'C:\Users\Owner\Downloads\sitting-posture-evaluation\cv\side_normal.mkv'`

In [ ]:
VIDEO_PATH = r'C:\Users\Owner\Downloads\sitting-posture-evaluation\cv\side_normal.mkv'
OUTPUT_DIR = Path('posture_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_file = Path(VIDEO_PATH)
if not video_file.exists():
    raise FileNotFoundError(f'Video file was not found: {video_file}\nUpdate VIDEO_PATH in this cell.')

video_stem = video_file.stem
ANNOTATED_PATH = OUTPUT_DIR / f'{video_stem}_annotated.mp4'
CSV_PATH = OUTPUT_DIR / f'{video_stem}_features.csv'

print('Input video:', video_file)
print('Annotated output:', ANNOTATED_PATH.resolve())
print('Feature CSV:', CSV_PATH.resolve())

## 4. Helper functions

The angles are geometric features, not posture classifications. This notebook records the features so you can inspect them and later define/test your neutral, slouching, and leaning-forward classes.

In [ ]:
def calculate_angle(a, b, c):
    """Return angle ABC in degrees, or NaN if coordinates are invalid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)
    if not (np.isfinite(a).all() and np.isfinite(b).all() and np.isfinite(c).all()):
        return float('nan')
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom == 0:
        return float('nan')
    cosine = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def get_xy(landmarks, landmark_id):
    """Return normalized x,y coordinates for a MediaPipe landmark."""
    lm = landmarks[landmark_id]
    return (lm.x, lm.y)


def visibility_of(landmarks, landmark_id):
    return float(landmarks[landmark_id].visibility)


def side_view_features(landmarks):
    """Extract side-view-friendly angles from MediaPipe Pose landmarks.

    Uses whichever side has greater combined shoulder/hip visibility.
    """
    L = mp_pose.PoseLandmark
    left_shoulder = L.LEFT_SHOULDER.value
    right_shoulder = L.RIGHT_SHOULDER.value
    left_elbow = L.LEFT_ELBOW.value
    right_elbow = L.RIGHT_ELBOW.value
    left_wrist = L.LEFT_WRIST.value
    right_wrist = L.RIGHT_WRIST.value
    left_hip = L.LEFT_HIP.value
    right_hip = L.RIGHT_HIP.value
    left_knee = L.LEFT_KNEE.value
    right_knee = L.RIGHT_KNEE.value
    left_ankle = L.LEFT_ANKLE.value
    right_ankle = L.RIGHT_ANKLE.value

    left_score = visibility_of(landmarks, left_shoulder) + visibility_of(landmarks, left_hip)
    right_score = visibility_of(landmarks, right_shoulder) + visibility_of(landmarks, right_hip)
    side = 'left' if left_score >= right_score else 'right'

    if side == 'left':
        shoulder, elbow, wrist = left_shoulder, left_elbow, left_wrist
        hip, knee, ankle = left_hip, left_knee, left_ankle
    else:
        shoulder, elbow, wrist = right_shoulder, right_elbow, right_wrist
        hip, knee, ankle = right_hip, right_knee, right_ankle

    nose = L.NOSE.value
    ear = L.LEFT_EAR.value if side == 'left' else L.RIGHT_EAR.value

    shoulder_xy = get_xy(landmarks, shoulder)
    elbow_xy = get_xy(landmarks, elbow)
    wrist_xy = get_xy(landmarks, wrist)
    hip_xy = get_xy(landmarks, hip)
    knee_xy = get_xy(landmarks, knee)
    ankle_xy = get_xy(landmarks, ankle)
    ear_xy = get_xy(landmarks, ear)
    nose_xy = get_xy(landmarks, nose)

    # Approximate torso inclination relative to vertical in the image plane.
    torso_dx = shoulder_xy[0] - hip_xy[0]
    torso_dy = hip_xy[1] - shoulder_xy[1]
    torso_tilt_from_vertical = float(np.degrees(np.arctan2(abs(torso_dx), abs(torso_dy)))) if abs(torso_dy) + abs(torso_dx) > 0 else float('nan')

    # Hip angle: shoulder-hip-knee. Shoulder angle: ear/shoulder/hip approximation.
    hip_angle = calculate_angle(shoulder_xy, hip_xy, knee_xy)
    shoulder_angle = calculate_angle(ear_xy, shoulder_xy, hip_xy)
    elbow_angle = calculate_angle(shoulder_xy, elbow_xy, wrist_xy)
    knee_angle = calculate_angle(hip_xy, knee_xy, ankle_xy)

    shoulder_vis = visibility_of(landmarks, shoulder)
    hip_vis = visibility_of(landmarks, hip)
    pose_reliable = (shoulder_vis >= 0.5 and hip_vis >= 0.5)

    return {
        'visible_side': side,
        'shoulder_angle_deg': shoulder_angle,
        'hip_angle_deg': hip_angle,
        'elbow_angle_deg': elbow_angle,
        'knee_angle_deg': knee_angle,
        'torso_tilt_from_vertical_deg': torso_tilt_from_vertical,
        'head_x': nose_xy[0],
        'head_y': nose_xy[1],
        'shoulder_x': shoulder_xy[0],
        'shoulder_y': shoulder_xy[1],
        'hip_x': hip_xy[0],
        'hip_y': hip_xy[1],
        'shoulder_visibility': shoulder_vis,
        'hip_visibility': hip_vis,
        'pose_reliable': bool(pose_reliable),
    }

print('Helper functions ready.')

## 5. Load YOLO and process the video

The script detects people with YOLO, estimates pose with MediaPipe, draws landmarks, and stores one CSV row per frame. The first run may download the `yolo11n.pt` model weights.

In [ ]:
MODEL_NAME = 'yolo11n.pt'
CONFIDENCE_THRESHOLD = 0.35

model = YOLO(MODEL_NAME)

cap = cv2.VideoCapture(str(video_file))
if not cap.isOpened():
    raise RuntimeError(f'OpenCV could not open the video: {video_file}. Check the path and whether this video codec is supported.')

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or not np.isfinite(fps) or fps <= 0:
    fps = 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(ANNOTATED_PATH), fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError('Could not create the output video. Check that the output folder is writable.')

records = []
frame_index = 0
pose_detector = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        timestamp_seconds = frame_index / fps
        annotated = frame.copy()
        row = {
            'video_file': video_file.name,
            'posture_label': video_file.stem,
            'frame_index': frame_index,
            'timestamp_seconds': timestamp_seconds,
            'person_detected': False,
            'pose_detected': False,
        }

        # YOLO class 0 is person for the standard COCO model.
        detections = model.predict(frame, classes=[0], conf=CONFIDENCE_THRESHOLD, verbose=False)
        best_box = None
        best_conf = -1.0
        if detections and detections[0].boxes is not None:
            for box in detections[0].boxes:
                conf = float(box.conf[0].cpu().item())
                if conf > best_conf:
                    best_conf = conf
                    best_box = box

        if best_box is not None:
            row['person_detected'] = True
            x1, y1, x2, y2 = [int(v) for v in best_box.xyxy[0].cpu().tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(width, x2), min(height, y2)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(annotated, f'Person {best_conf:.2f}', (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)

            # Estimate pose on the full frame to keep normalized coordinates consistent.
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pose_result = pose_detector.process(rgb)
            if pose_result.pose_landmarks:
                row['pose_detected'] = True
                mp_drawing.draw_landmarks(
                    annotated,
                    pose_result.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                )
                row.update(side_view_features(pose_result.pose_landmarks.landmark))
            else:
                row.update({
                    'visible_side': None,
                    'shoulder_angle_deg': np.nan,
                    'hip_angle_deg': np.nan,
                    'elbow_angle_deg': np.nan,
                    'knee_angle_deg': np.nan,
                    'torso_tilt_from_vertical_deg': np.nan,
                    'head_x': np.nan, 'head_y': np.nan,
                    'shoulder_x': np.nan, 'shoulder_y': np.nan,
                    'hip_x': np.nan, 'hip_y': np.nan,
                    'shoulder_visibility': np.nan,
                    'hip_visibility': np.nan,
                    'pose_reliable': False,
                })
        else:
            row.update({
                'visible_side': None,
                'shoulder_angle_deg': np.nan,
                'hip_angle_deg': np.nan,
                'elbow_angle_deg': np.nan,
                'knee_angle_deg': np.nan,
                'torso_tilt_from_vertical_deg': np.nan,
                'head_x': np.nan, 'head_y': np.nan,
                'shoulder_x': np.nan, 'shoulder_y': np.nan,
                'hip_x': np.nan, 'hip_y': np.nan,
                'shoulder_visibility': np.nan,
                'hip_visibility': np.nan,
                'pose_reliable': False,
            })

        cv2.putText(annotated, f'Frame {frame_index}  Time {timestamp_seconds:.2f}s', (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        writer.write(annotated)
        records.append(row)
        frame_index += 1

        if frame_index % 50 == 0:
            print(f'Processed {frame_index}/{frame_count if frame_count > 0 else "?"} frames')
finally:
    cap.release()
    writer.release()
    pose_detector.close()

features_df = pd.DataFrame(records)
features_df.to_csv(CSV_PATH, index=False)

print(f'Finished. Processed {frame_index} frames.')
print('Annotated video:', ANNOTATED_PATH.resolve())
print('Feature CSV:', CSV_PATH.resolve())
features_df.head()

## 6. Quick quality check

This summarizes whether a person and pose were detected. A high number of unreliable/missing poses can indicate that the person is too far away, partly outside the frame, or poorly lit.

In [ ]:
if not features_df.empty:
    print('Frames:', len(features_df))
    print('Person detected:', int(features_df['person_detected'].fillna(False).sum()))
    print('Pose detected:', int(features_df['pose_detected'].fillna(False).sum()))
    if 'pose_reliable' in features_df.columns:
        print('Reliable pose frames:', int(features_df['pose_reliable'].fillna(False).sum()))
    display(features_df[['frame_index', 'timestamp_seconds', 'person_detected', 'pose_detected', 'pose_reliable', 'shoulder_angle_deg', 'hip_angle_deg', 'torso_tilt_from_vertical_deg']].head(10))
else:
    print('No frames were processed. Check the video path and codec.')

## Notes for your thesis

- `posture_label` is currently taken from the video filename. For example, `side_normal.mkv` becomes `side_normal`; rename the video or edit the label in the configuration if you want standardized labels.
- This notebook **extracts features; it does not yet train a classifier** or automatically decide whether a frame is neutral, slouching, or leaning forward.
- Use the same camera position, distance, and approximate framing for your initial side-view recordings. Keep pilot recordings separate from your final evaluation dataset.
- `torso_tilt_from_vertical_deg` is an image-plane approximation. Review sample annotated frames and the CSV before treating any feature as a reliable scientific measurement.
- Outputs are saved in the `posture_output` folder relative to the notebook's current working directory.

In [ ]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Owner\AppData\Local\Programs\Python\Python311\python.exe


ImportError: DLL load failed while importing _framework_bindings: A dynamic link library (DLL) initialization routine failed.